# DataFeed & MarketData — Esempio interattivo

Mostra tutte le funzionalità di `engine/datafeed.py`:
- **`DataFeed`** — I/O su parquet: caricamento eager, lazy scan, export Excel
- **`MarketData`** — accesso bar-by-bar con offset (T, T-1, T-N, slice) e protezione da lookahead bias

In [1]:
import sys
from pathlib import Path
from datetime import date

import polars as pl
import plotly.graph_objects as go
from plotly.subplots import make_subplots

sys.path.insert(0, str(Path.cwd().parent))
from engine.datafeed import DataFeed, MarketData

PRICES_BASE = str(Path.cwd().parent / "database")

In [2]:
# ── Configurazione ─────────────────────────────────────────
TICKER = "AAPL"
START  = "2020-01-01"
END    = str(date.today())
FREQ   = "eod"   # eod | 1m | 5m | 1h | 4h
# ───────────────────────────────────────────────────────────

---
## 1. `DataFeed.get_market_data()` — caricamento eager

Carica tutti i dati in memoria come `pl.DataFrame`.

In [5]:
feed = DataFeed(prices_base=PRICES_BASE)
df = feed.get_market_data(START, END, FREQ, [TICKER])

print(f"Righe totali : {df.height}")
print(f"Range date   : {df['date'].min()} -> {df['date'].max()}")

Righe totali : 1607
Range date   : 2020-01-02 -> 2026-05-26


In [6]:
# Ultime 5 righe
df.tail(5)

date,time,ticker,open,high,low,close,volume,insertion_time,type
date,time,str,f64,f64,f64,f64,f64,datetime[μs],str
2026-05-19,02:00:00,"""AAPL""",296.97,300.51,296.35,298.97,1.9673981e7,2026-05-26 22:14:00,"""eod"""
2026-05-20,02:00:00,"""AAPL""",298.18,302.8,298.08,302.25,2.4179314e7,2026-05-26 22:14:00,"""eod"""
2026-05-21,02:00:00,"""AAPL""",301.03,305.54,300.4,304.99,2.1166315e7,2026-05-26 22:14:00,"""eod"""
2026-05-22,02:00:00,"""AAPL""",306.06,311.4,305.84,308.82,2.4956356e7,2026-05-26 22:14:00,"""eod"""
2026-05-26,02:00:00,"""AAPL""",309.61,311.82,307.67,308.33,2.4448064e7,2026-05-26 22:17:00,"""eod"""


### Grafico OHLCV

In [7]:
dates  = df["date"].to_list()
opens  = df["open"].to_list()
highs  = df["high"].to_list()
lows   = df["low"].to_list()
closes = df["close"].to_list()
vols   = df["volume"].to_list()

fig = make_subplots(
    rows=2, cols=1, shared_xaxes=True,
    row_heights=[0.75, 0.25], vertical_spacing=0.03,
)
fig.add_trace(
    go.Candlestick(
        x=dates, open=opens, high=highs, low=lows, close=closes,
        name=TICKER,
        increasing_line_color="#26a69a", decreasing_line_color="#ef5350",
    ),
    row=1, col=1,
)
colors = ["#26a69a" if c >= o else "#ef5350" for c, o in zip(closes, opens)]
fig.add_trace(
    go.Bar(x=dates, y=vols, name="Volume", marker_color=colors, opacity=0.7),
    row=2, col=1,
)
fig.update_layout(
    title=f"{TICKER} — {FREQ.upper()} ({START} / {END})",
    xaxis_rangeslider_visible=False,
    template="plotly_dark",
    height=600,
    margin=dict(t=50, b=20),
)
fig.update_yaxes(title_text="Prezzo", row=1, col=1)
fig.update_yaxes(title_text="Volume", row=2, col=1)
fig.show()

---
## 2. `DataFeed.scan_prices()` — lazy scan

Ritorna un `pl.LazyFrame`. I dati non vengono caricati finché non si chiama `.collect()`. Utile per filtrare su dataset grandi prima di materializzare.

In [8]:
lf = feed.scan_prices(START, END, FREQ, [TICKER])
print(f"Tipo: {type(lf).__name__}  (nessun dato in RAM finché non si colleziona)")

# Esempio: giorni con volume sopra la media — filtro lazy
avg_vol = df["volume"].mean()
high_vol = (
    lf.filter(pl.col("volume") > avg_vol)
      .select(["date", "close", "volume"])
      .collect(engine="streaming")
)
print(f"Barre con volume > media ({avg_vol:,.0f}): {high_vol.height}")
high_vol.tail(5)

Tipo: LazyFrame  (nessun dato in RAM finché non si colleziona)
Barre con volume > media (56,446,809): 626


date,close,volume
date,f64,f64
2025-08-06,213.25,6.8984952e7
2025-08-07,220.03,6.0501283e7
2025-08-08,229.35,8.1284112e7
2025-09-19,245.5,6.5353497e7
2025-09-22,256.08,6.7874548e7


---
## 3. `DataFeed.get_data_excel()` — export Excel

In [10]:
output_path = str(Path.cwd() / f"{TICKER}_{START}_{END}_{FREQ}.xlsx")
# path = feed.get_data_excel(START, END, FREQ, [TICKER], output_path=output_path)
# print(f"File Excel creato: {path}")

---
## 4. `MarketData` — accesso bar-by-bar

Layer usato dall'engine di backtest. Espone `price()`, `get()`, `ohlc()` con offset temporali.

In [11]:
md = MarketData.from_datafeed(feed, START, END, FREQ, [TICKER])

print(f"Simboli       : {md.symbols()}")
print(f"Data corrente : {md.current_date()}")
print(f"Ha {TICKER}?  : {md.has_symbol(TICKER)}")

Simboli       : ['AAPL']
Data corrente : 2026-05-26
Ha AAPL?  : True


In [12]:
# OHLCV con offset T, T-1, T-2, T-3, T-4
rows = []
for n in range(5):
    ohlc = md.ohlc(TICKER, n)
    vol  = md.volume(TICKER, n)
    rows.append({"bar": f"T-{n}" if n else "T", **ohlc, "volume": vol})

pl.DataFrame(rows)

bar,open,high,low,close,volume
str,f64,f64,f64,f64,f64
"""T""",309.61,311.82,307.67,308.33,2.4448064e7
"""T-1""",306.06,311.4,305.84,308.82,2.4956356e7
"""T-2""",301.03,305.54,300.4,304.99,2.1166315e7
"""T-3""",298.18,302.8,298.08,302.25,2.4179314e7
"""T-4""",296.97,300.51,296.35,298.97,1.9673981e7


In [13]:
# Ultimi 10 bar completi come DataFrame
md.get(TICKER, slice(None, 10))

date,time,ticker,open,high,low,close,volume,insertion_time,type
date,time,str,f64,f64,f64,f64,f64,datetime[μs],str
2026-05-12,02:00:00,"""AAPL""",292.51,295.27,292.51,294.8,2.2461822e7,2026-05-26 22:14:00,"""eod"""
2026-05-13,02:00:00,"""AAPL""",293.41,300.92,293.41,298.87,2.887998e7,2026-05-26 22:14:00,"""eod"""
2026-05-14,02:00:00,"""AAPL""",299.82,300.45,295.38,298.21,2.0829566e7,2026-05-26 22:14:00,"""eod"""
2026-05-15,02:00:00,"""AAPL""",297.78,303.2,296.52,300.23,3.0112207e7,2026-05-26 22:14:00,"""eod"""
2026-05-18,02:00:00,"""AAPL""",300.24,300.65,294.91,297.84,1.9529793e7,2026-05-26 22:14:00,"""eod"""
2026-05-19,02:00:00,"""AAPL""",296.97,300.51,296.35,298.97,1.9673981e7,2026-05-26 22:14:00,"""eod"""
2026-05-20,02:00:00,"""AAPL""",298.18,302.8,298.08,302.25,2.4179314e7,2026-05-26 22:14:00,"""eod"""
2026-05-21,02:00:00,"""AAPL""",301.03,305.54,300.4,304.99,2.1166315e7,2026-05-26 22:14:00,"""eod"""
2026-05-22,02:00:00,"""AAPL""",306.06,311.4,305.84,308.82,2.4956356e7,2026-05-26 22:14:00,"""eod"""


### Close degli ultimi 60 bar (via `price()` + slice)

In [14]:
N = 60
close_series = md.price(TICKER, "close", slice(None, N))
last_n_dates = md.get(TICKER, slice(None, N))["date"].to_list()

fig2 = go.Figure(go.Scatter(
    x=last_n_dates, y=close_series.to_list(),
    mode="lines+markers",
    line=dict(color="#29b6f6", width=2),
    marker=dict(size=4),
    name="Close",
))
fig2.update_layout(
    title=f"{TICKER} — ultimi {N} close  (MarketData.price slice)",
    xaxis_title="Data", yaxis_title="Close",
    template="plotly_dark", height=400,
)
fig2.show()

---
## 5. Protezione lookahead bias — `allow_lookahead`

Per default offset negativi (accesso al futuro) sollevano `ValueError`.  
Passare `allow_lookahead=True` per abilitarli in contesti di research.

In [15]:
# Default — deve sollevare ValueError
try:
    md.price(TICKER, "close", -1)
except ValueError as e:
    print(f"OK — ValueError:\n  {e}")

OK — ValueError:
  Negative offset (-1) accesses future bars and introduces lookahead bias. Set allow_lookahead=True on MarketData to allow this in research contexts.


In [16]:
# Research mode: allow_lookahead=True (es. forward return per labeling)
md_research = MarketData.from_datafeed(
    feed, START, END, FREQ, [TICKER], allow_lookahead=True
)

close_t0  = md_research.price(TICKER, "close", 0)
close_tp1 = md_research.price(TICKER, "close", -1)

if close_tp1 is not None:
    fwd_ret = (close_tp1 - close_t0) / close_t0
    print(f"Close T    : {close_t0:.2f}")
    print(f"Close T+1  : {close_tp1:.2f}")
    print(f"Forward ret: {fwd_ret:+.4f}")
else:
    print("T+1 non disponibile (siamo all'ultima barra)")

T+1 non disponibile (siamo all'ultima barra)
